# 1. EDA — Biohub Cell Tracking During Development

Purpose: understand the OME-Zarr + GEFF data format, survey shape/scale/
ground-truth density across multiple training videos (not just one), and
inspect a single video in depth (frame visualization, division timing)
before designing/tuning a model.

Runs on Kaggle via `scripts/push_kaggle_kernel.sh eda` (the competition
mount and the public `cellmot-baseline-artifacts` dataset, which bundles a
working `repo/`, both auto-detect — see the Setup cell), or locally after
downloading one sample and pointing `$CELLMOT_DATA_DIR` at it. Last
completed run: 2026-07-21 (kernel `tuannm3812/biohub-eda`) — results below
and in `docs/2_eda_insights.md`.

## 1. Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()


def _find_mount(candidates: list[Path], marker: str) -> Path | None:
    """Return the first candidate containing ``marker``, else scan /kaggle/input."""
    for c in candidates:
        if (c / marker).exists():
            return c
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in kaggle_input.glob(f"**/{marker}"):
            return p.parent
    return None


if IS_KAGGLE:
    # Pin numpy/scipy/torch to whatever Kaggle's base image already has:
    # letting pip pick a newer numpy for zarr>=3.0.10 breaks the base
    # image's precompiled scipy (ImportError deep in scipy.spatial/
    # numpy._core -- an ABI mismatch between new numpy and the untouched
    # old scipy build). tracksdata also depends on torch; pin it too so an
    # unpinned reinstall can't swap the base image's build for one that
    # doesn't match its GPU driver (irrelevant with GPU off here, but keeps
    # this cell identical to 02_baseline_modeling.ipynb's). Both issues
    # observed directly on this competition's Kaggle image. Must also run
    # before importing numpy/matplotlib below, for the same reason.
    import importlib.metadata

    _pinned = {
        pkg: importlib.metadata.version(pkg) for pkg in ("numpy", "scipy", "torch")
    }
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            *(f"{pkg}=={ver}" for pkg, ver in _pinned.items()),
            "zarr>=3.0.10", "tqdm", "polars",
            "tracksdata @ git+https://github.com/royerlab/tracksdata@main",
        ],
        check=True,
    )
    ARTIFACTS_MOUNT = _find_mount(
        [
            Path("/kaggle/input/cellmot-baseline-artifacts"),
            Path("/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        ],
        "weights",
    )
    if ARTIFACTS_MOUNT is None:
        raise FileNotFoundError(
            "cellmot-baseline-artifacts dataset not found under /kaggle/input -- add "
            "it as a data source (see kernel-metadata.json)."
        )
    REPO_ROOT = ARTIFACTS_MOUNT / "repo"  # read-only is fine -- EDA never writes predictions/weights
else:
    REPO_ROOT = Path.cwd().parent

import matplotlib.pyplot as plt  # noqa: E402 -- see the pip-install note above
import numpy as np  # noqa: E402

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from dataspec import DATASET_PATH  # noqa: E402 -- needs REPO_ROOT on sys.path first

from tracking_cellmot.io import list_datasets, open_dataset  # noqa: E402

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR = Path(f"/kaggle/input/competitions/{COMPETITION}")
TEST_DIR = COMP_DIR / "test" if IS_KAGGLE else REPO_ROOT / "data" / "test"

SEED = 0
np.random.seed(SEED)

DATASET_NAME = None      # None -> pick the first available training dataset
FRAME_INDEX = 0           # timepoint to visualize for the single-dataset deep dive
N_VIDEOS_FOR_STATS = 10   # how many train videos to summarize in the stats table; None = all

print(f"IS_KAGGLE={IS_KAGGLE}  REPO_ROOT={REPO_ROOT}  DATASET_PATH={DATASET_PATH}  TEST_DIR={TEST_DIR}")

## 2. List available datasets (train + test)

In [ ]:
train_datasets = list_datasets(DATASET_PATH, require_geff=True)
print(f"DATASET_PATH = {DATASET_PATH}")
print(f"{len(train_datasets)} train datasets with ground truth found")
for p in train_datasets[:10]:
    print(" -", p.stem)

test_datasets = list_datasets(TEST_DIR, require_geff=False) if TEST_DIR.exists() else []
print(f"\nTEST_DIR = {TEST_DIR}")
print(f"{len(test_datasets)} test datasets found (no ground truth, per the competition)")
for p in test_datasets[:10]:
    print(" -", p.stem)

## 3. Multi-video stats table (train)

A single video isn't representative — summarize shape, scale, and
ground-truth density across the first `N_VIDEOS_FOR_STATS` train videos so
modeling decisions (e.g. `--unet-batch-size`, `--det-threshold` in
`02_baseline_modeling.ipynb`) are grounded across the dataset, not one clip.

In [ ]:
import polars as pl

survey = train_datasets[:N_VIDEOS_FOR_STATS] if N_VIDEOS_FOR_STATS else train_datasets

rows = []
for p in survey:
    ds_i = open_dataset(p, normalize=False, require_tracks=True)
    out_degree_i = ds_i.tracks.edge_attrs().group_by("source_id").len()
    n_t = ds_i.image.shape[0]
    n_nodes = ds_i.tracks.num_nodes()
    rows.append({
        "dataset": p.stem,
        "T": n_t,
        "Z": ds_i.image.shape[1],
        "Y": ds_i.image.shape[2],
        "X": ds_i.image.shape[3],
        "dtype": str(ds_i.image.dtype),
        "scale_z_y_x_um": tuple(round(s, 4) for s in ds_i.scale),
        "nodes": n_nodes,
        "edges": ds_i.tracks.num_edges(),
        "divisions": int((out_degree_i["len"] == 2).sum()),
        "nodes_per_timepoint": round(n_nodes / n_t, 2),
    })

stats = pl.DataFrame(rows)
print(f"Surveyed {len(survey)}/{len(train_datasets)} train videos")
stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(stats["dataset"], stats["nodes_per_timepoint"], color=plt.cm.viridis(0.5))
axes[0].set_title("Annotated nodes per timepoint")
axes[0].tick_params(axis="x", rotation=90)
axes[1].bar(stats["dataset"], stats["divisions"], color=plt.cm.viridis(0.8))
axes[1].set_title("Division events")
axes[1].tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

*Insight: the stats table/plot themselves render on Kaggle's kernel page
(kaggle.com/code/tuannm3812/biohub-eda) — the CLI-downloadable log only
captures `print()` stdout, not DataFrame/plot rendering, so the per-video
breakdown isn't reproduced here. Open question worth checking directly on
that page: how much do shape (T/Z/Y/X) and annotation density vary across
the 10 surveyed videos — enough to need a per-video `--det-threshold`
rather than one global value?*

## 4. Count calibration signal: `estimated_number_of_nodes`

Per `docs/3_strategy.md` (synthesized from public reference notebooks):
each `.geff`'s `zarr.json` metadata carries
`attributes.geff.extra.estimated_number_of_nodes` — an estimate of the
*true* cell count, independent of the sparse GT annotation count above.
Several top public submissions use the ratio of this estimate to their own
detection count to calibrate a per-video detection budget (avoiding the
Adjusted Edge Jaccard's over-prediction penalty — `docs/metrics.md`). Not
yet used anywhere in our own pipeline; this cell just measures it.

In [ ]:
import json as _json


def read_estimated_nodes(geff_path: Path) -> int | None:
    """Read attributes.geff.extra.estimated_number_of_nodes from a .geff's zarr.json, if present."""
    meta_path = geff_path.with_suffix(".geff") / "zarr.json"
    if not meta_path.exists():
        return None
    try:
        meta = _json.loads(meta_path.read_text())
    except (OSError, ValueError):
        return None
    return meta.get("attributes", {}).get("geff", {}).get("extra", {}).get("estimated_number_of_nodes")


calib_rows = []
for p, row in zip(survey, stats.iter_rows(named=True), strict=True):
    est = read_estimated_nodes(p)
    calib_rows.append({
        "dataset": row["dataset"],
        "annotated_nodes": row["nodes"],
        "estimated_number_of_nodes": est,
        "annotated_fraction": round(row["nodes"] / est, 4) if est else None,
    })

calib = pl.DataFrame(calib_rows)
n_with_estimate = calib["estimated_number_of_nodes"].is_not_null().sum()
print(f"{n_with_estimate}/{len(calib)} surveyed videos have an estimated_number_of_nodes field")
calib

*Insight (real run, 2026-07-21): `estimated_number_of_nodes` is present on
**10/10** surveyed train videos — the field exists and is usable.
`annotated_fraction` values themselves are in the `calib` table above
(rendered on Kaggle's kernel page, not captured in the CLI log — same
limitation as section 3). Not yet checked: whether this field is **also**
present on the real `test/` videos, which don't have `.geff` ground truth —
required before building a `DET_THRESHOLD` calibration step around it
(`02_baseline_modeling.ipynb`'s roadmap step 2, `docs/3_strategy.md`).*

## 5. Deep dive: one dataset in detail

In [ ]:
name = DATASET_NAME or train_datasets[0].stem
ds = open_dataset(DATASET_PATH / name, normalize=False, require_tracks=True)

print(f"dataset:      {name}")
print(f"image shape:  {ds.image.shape}  (T, Z, Y, X)")
print(f"image dtype:  {ds.image.dtype}")
print(f"voxel scale:  {ds.scale}  microns (Z, Y, X)")
print(f"track nodes:  {ds.tracks.num_nodes()}")
print(f"track edges:  {ds.tracks.num_edges()}")

*Insight (real run, 2026-07-21): `44b6_0113de3b` has 100 timepoints but
only 52 annotated nodes / 50 edges total — ~0.5 nodes per timepoint on a
`(64, 256, 256)` volume. Confirms the GT is genuinely sparse (far fewer
annotations than visible cells), matching the competition's own framing —
see `docs/2_eda_insights.md` for the full writeup.*

### Visualize one frame with annotated cell centers

In [ ]:
frame = np.asarray(ds.image[FRAME_INDEX])
mip = frame.max(axis=0)  # max-intensity projection over Z

node_attrs = ds.tracks.node_attrs()
frame_nodes = node_attrs.filter(node_attrs["t"] == FRAME_INDEX)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(mip, cmap="viridis")
ax.scatter(frame_nodes["x"], frame_nodes["y"], s=12, facecolors="none", edgecolors="white", linewidths=0.8)
ax.set_title(f"{name} — frame {FRAME_INDEX} — {frame_nodes.height} annotated cells")
ax.axis("off")
plt.show()

*Insight: ran successfully 2026-07-21, but the rendered image itself isn't
captured by the CLI log (same limitation noted in section 3) — view it
directly on kaggle.com/code/tuannm3812/biohub-eda to check whether
annotated centers (white circles) actually land on visible cells rather
than background, which would flag a scale/orientation bug worth catching
before trusting anything downstream.*

### Division timing (this dataset)

In [ ]:
edge_attrs = ds.tracks.edge_attrs()
out_degree = edge_attrs.group_by("source_id").len()
n_divisions = (out_degree["len"] == 2).sum()
print(f"division events (source with 2 outgoing edges): {n_divisions}")
print(f"total timepoints: {ds.image.shape[0]}")

*Insight (real run, 2026-07-21): 0 divisions in `44b6_0113de3b`'s 100
timepoints — consistent with `02_baseline_modeling.ipynb`'s validation
finding that this checkpoint currently gets 0 division_jaccard credit.
One video isn't enough to conclude divisions are simply rare in this
dataset rather than under-annotated or under-detected — worth checking a
handful more videos (raise `N_VIDEOS_FOR_STATS` in section 3, or set
`DATASET_NAME` here to a few specific stems) before concluding division
recovery isn't worth pursuing.*

## 6. Findings / limitations / next experiment

- **Findings** (real runs, 2026-07-21): 199 train videos with ground
  truth, only **4 test videos** — confirmed via section 2's `list_datasets`
  call against the real competition mount
  (`/kaggle/input/competitions/.../test`, not the artifacts dataset), so
  this is the actual held-out test set, not a partial slice. GT is
  genuinely sparse (sample video: 100 timepoints, 52 nodes / 50 edges, 0
  divisions). `estimated_number_of_nodes` exists on 10/10 surveyed train
  videos, so it's usable for count calibration — pending confirmation on
  `test/`. Full writeup: `docs/2_eda_insights.md`.
- **Limitations**: `N_VIDEOS_FOR_STATS` caps the multi-video survey (default
  10, of 199 found) for a fast first pass — rerun with `None` for the full
  dataset now that the notebook's Kaggle runtime is known to be fast. The
  rich stats table/plots/frame visualization only render on Kaggle's
  kernel page, not in the CLI-downloadable log — review them there.
- **Next**: check `estimated_number_of_nodes` against `test/` videos, then
  wire the resulting calibration into `02_baseline_modeling.ipynb`'s
  `DET_THRESHOLD`. See `docs/3_strategy.md` for the full prioritized
  roadmap, synthesized from six public reference notebooks — currently
  centered on graph-repair post-processing, which our own validation found
  *regresses* the score as first implemented (see
  `02_baseline_modeling.ipynb` section 2) — not architecture changes.